# Lab 01 — AI_COMPLETE Basics

`AI_COMPLETE` is the foundational function for calling LLMs inside Snowflake SQL.
This lab covers the core patterns you'll use in every other lab.

| Item | Detail |
|---|---|
| **Function** | `AI_COMPLETE` |
| **Models** | Claude (`claude-haiku-4-5`, `claude-sonnet-4-5`, `claude-opus-4-5`), Llama (`llama3.1-8b`, `llama3.1-70b`, `llama3.3-70b`, `llama4-maverick`), Mistral (`mistral-large2`, `mistral-medium-3`), OpenAI (`openai-gpt-4.1`, `openai-o4-mini`), Google (`gemini-2.5-flash`), Snowflake (`snowflake-arctic`), DeepSeek (`deepseek-r1`) and more — run `SHOW MODELS IN SNOWFLAKE.MODELS` for the full list |
| **Exam Domain** | 1.0 Gen AI Overview, 2.0 Gen AI & LLM Functions |
| **You'll learn** | Simple prompts, system/user messages, options, prompt patterns, model comparison |


## Step 1 — Environment Setup

> **AI SQL Functions** — This lab uses the preferred `AI_` prefixed functions. 
Equivalent legacy functions: `AI_COMPLETE` (replaces `SNOWFLAKE.CORTEX.COMPLETE`).

## Available Models for AI_COMPLETE

`AI_COMPLETE` supports a wide range of LLMs. Run `SHOW MODELS IN SNOWFLAKE.MODELS` to see all models available in your account.

| Provider | Models | Best for |
|---|---|---|
| **Anthropic** | `claude-haiku-4-5` · `claude-sonnet-4-5` · `claude-sonnet-4-6` · `claude-opus-4-5` · `claude-opus-4-6` | General purpose, instruction following, low latency (Haiku) |
| **Meta (Llama)** | `llama3.1-8b` · `llama3.1-70b` · `llama3.1-405b` · `llama3.3-70b` · `llama4-maverick` · `llama4-scout` | Open-weight, cost-effective, wide availability |
| **Mistral** | `mistral-large2` · `mistral-medium-3` · `mistral-7b` · `mixtral-8x7b` | Multilingual, coding, concise outputs |
| **OpenAI** | `openai-gpt-4.1` · `openai-o4-mini` · `openai-gpt-5` | Complex reasoning, structured outputs |
| **Google** | `gemini-2.5-flash` · `gemini-2.5-flash-lite` · `gemini-3-pro` | Multimodal, long context |
| **Snowflake** | `snowflake-arctic` · `snowflake-llama-3.3-70b` | Snowflake-optimised, SQL-focused |
| **DeepSeek** | `deepseek-r1` | Chain-of-thought reasoning |

> **Tip:** Use `claude-haiku-4-5` or `llama3.1-8b` for iterative lab work (fast + low cost). Use `claude-sonnet-4-5` or `llama3.3-70b` for higher quality outputs.

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Basic Completion

The simplest pattern: model name + prompt string.

In [ ]:
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    'Explain data warehousing in exactly three sentences.'
) AS response;

## Step 3 — System + User Message Pattern

Use the structured message array for role-based prompting. The `system` message sets behavior; the `user` message is the question.

In [ ]:
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    [
        {'role': 'system', 'content': 'You are a concise Snowflake expert. Answer in bullet points.'},
        {'role': 'user', 'content': 'What are the key benefits of Snowflake Cortex AI functions?'}
    ],
    {'temperature': 0.3, 'max_tokens': 500}
):choices[0]:messages::STRING AS response;

## Step 3b — Multi-Turn Conversation (assistant role)

The `assistant` role passes prior model responses back into the conversation, enabling stateful multi-turn dialogue. The three roles are:

| Role | Purpose |
|---|---|
| `system` | Background instructions and persona (optional, must be first) |
| `user` | Human prompts |
| `assistant` | Prior model responses fed back for context |

In [ ]:
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    [
        {'role': 'system',    'content': 'You are a Snowflake architecture advisor. Be concise.'},
        {'role': 'user',      'content': 'What is a virtual warehouse?'},
        {'role': 'assistant', 'content': 'A virtual warehouse is a named cluster of compute resources that executes queries and DML operations.'},
        {'role': 'user',      'content': 'How does it differ from a database?'}
    ],
    {}
):choices[0]:messages::STRING AS multi_turn_response;

## Step 4 — AI_COMPLETE Options Reference

The third argument controls LLM behavior:

| Option | Type | Description |
|---|---|---|
| `temperature` | FLOAT 0.0–1.0 | Controls randomness. 0 = deterministic. |
| `max_tokens` | INT | Maximum output length in tokens. |
| `top_p` | FLOAT 0.0–1.0 | Nucleus sampling threshold. |
| `guardrails` | BOOLEAN | Enable Cortex Guard safety screening. |
| `response_format` | OBJECT | Force output format, e.g. `{type: 'json'}`. |


In [ ]:
SELECT
    'deterministic' AS mode,
    AI_COMPLETE(
        'claude-haiku-4-5',
        [{'role': 'user', 'content': 'Name 3 Snowflake features.'}],
        {'temperature': 0.0, 'max_tokens': 100}
    ):choices[0]:messages::STRING AS response
UNION ALL
SELECT
    'creative',
    AI_COMPLETE(
        'claude-haiku-4-5',
        [{'role': 'user', 'content': 'Name 3 Snowflake features.'}],
        {'temperature': 0.9, 'max_tokens': 100}
    ):choices[0]:messages::STRING;

## Step 4b — Additional Options: top_p & Usage Metadata

Beyond `temperature` and `max_tokens`, the options object supports:
- **`top_p`** — nucleus sampling (alternative to temperature); restricts token selection to a cumulative probability
- **Usage metadata** — the response object includes token counts in the `usage` key

In [ ]:
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    [{'role': 'user', 'content': 'Suggest a creative name for a data pipeline.'}],
    {'top_p': 0.9, 'max_tokens': 50}
):choices[0]:messages::STRING AS top_p_response;

In [ ]:
SELECT
    resp:choices[0]:messages::STRING AS response,
    resp:usage:prompt_tokens::INT AS prompt_tokens,
    resp:usage:completion_tokens::INT AS completion_tokens,
    resp:usage:total_tokens::INT AS total_tokens,
    resp:model::STRING AS model_used
FROM (
    SELECT AI_COMPLETE(
        'claude-haiku-4-5',
        [{'role': 'user', 'content': 'What is Snowpark?'}],
        {'max_tokens': 100}
    ) AS resp
);

## Step 5 — Prompt Engineering Patterns

Five patterns you'll use constantly: zero-shot, few-shot, chain-of-thought, extraction, summarization.

In [ ]:
CREATE OR REPLACE TABLE prompt_experiments (
    experiment_id INT AUTOINCREMENT,
    pattern_name  VARCHAR(100),
    prompt_text   TEXT
);

INSERT INTO prompt_experiments (pattern_name, prompt_text) VALUES
('zero_shot', 'Classify this text as POSITIVE, NEGATIVE, or NEUTRAL: The new feature update broke my workflow completely.'),
('few_shot', 'Classify sentiment. Examples:\nInput: Love it! -> POSITIVE\nInput: Terrible. -> NEGATIVE\nInput: It is okay. -> NEUTRAL\n\nInput: The new feature update broke my workflow completely. ->'),
('chain_of_thought', 'Analyze the sentiment step by step:\n1. Identify key phrases\n2. Determine emotional tone\n3. Give final classification\n\nText: The new feature update broke my workflow completely.'),
('extraction', 'Extract the following from this support ticket in JSON format: {issue_type, severity, product_area}.\nTicket: The dashboard export feature has been timing out for the past 3 days. This is blocking our weekly executive report.'),
('summarization', 'Summarize the key technical decisions in this meeting note in exactly 3 bullet points:\nWe discussed migrating from PostgreSQL to Snowflake. Team agreed on a phased approach starting with analytics workloads. Data engineering will own the pipeline migration while the BI team handles dashboard conversion. Target completion is Q2.');

In [ ]:
SELECT
    pattern_name,
    AI_COMPLETE('claude-haiku-4-5', prompt_text) AS llm_response
FROM prompt_experiments
ORDER BY experiment_id;

## Step 6 — Compare Models

Different models have different strengths. Compare on your specific task.

In [ ]:
SELECT
    'claude-haiku-4-5' AS model,
    AI_COMPLETE('claude-haiku-4-5', 'In one sentence, what makes Snowflake unique?') AS response
UNION ALL
SELECT
    'mistral-large2',
    AI_COMPLETE('mistral-large2', 'In one sentence, what makes Snowflake unique?')
UNION ALL
SELECT
    'llama3.1-70b',
    AI_COMPLETE('llama3.1-70b', 'In one sentence, what makes Snowflake unique?');

## Key Takeaways

- `AI_COMPLETE` supports both simple string prompts and structured message arrays.
- Use `temperature` (0.0–1.0) to control creativity; lower = more deterministic.
- Five key patterns: zero-shot, few-shot, chain-of-thought, extraction, summarization.
- Apply to table columns with SELECT for batch processing at scale.
- Compare models on your specific task — accuracy, latency, and cost vary.
